In [ ]:
import json
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split


# =============================================================================
# Global configuration
# =============================================================================
RANDOM_STATE = 42
POSITIVE_CLASS = 1
NEGATIVE_CLASS = 0
POSITIVE_CLASS_NAME = "HTS"
NEGATIVE_CLASS_NAME = "LTS"
TOP_N_FEATURES = 10

HTS_COLOR = "#d88f91"
LTS_COLOR = "#80bcc8"

plt.rcParams["figure.dpi"] = 150
warnings.filterwarnings("default")


# =============================================================================
# File and plotting utilities
# =============================================================================
def get_code_directory() -> Path:
    """Return the script directory, or the current directory in a notebook."""
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


def ensure_directory(path: Path) -> None:
    """Create a directory and any missing parent directories."""
    path.mkdir(parents=True, exist_ok=True)


def save_svg(fig: plt.Figure, save_path: Path) -> None:
    """Save a Matplotlib figure as SVG and close it."""
    save_path = save_path.with_suffix(".svg")
    fig.savefig(
        save_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


# =============================================================================
# Data preparation
# =============================================================================
def load_expression_data(data_path: Path) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """
    Load the expression matrix.

    The input file must contain a binary column named ``label`` with
    0 representing LTS and 1 representing HTS. All other retained columns
    are treated as gene-expression features.
    """
    if not data_path.exists():
        raise FileNotFoundError(
            f"Data file not found: {data_path}\n"
            "Place df_expr.csv in the same directory as this script or notebook."
        )

    dataframe = pd.read_csv(data_path)

    if "label" not in dataframe.columns:
        raise ValueError("The input data must contain a column named 'label'.")

    non_feature_columns = ["label"]

    # Treat a nonnumeric first column as a row or cell identifier.
    first_column = dataframe.columns[0]
    if first_column != "label" and not pd.api.types.is_numeric_dtype(
        dataframe[first_column]
    ):
        non_feature_columns.append(first_column)
        print(f"[INFO] Excluding nonnumeric identifier column: {first_column}")

    feature_columns = [
        column for column in dataframe.columns if column not in non_feature_columns
    ]

    if not feature_columns:
        raise ValueError("No numeric feature columns were found in df_expr.csv.")

    feature_matrix = dataframe[feature_columns].apply(
        pd.to_numeric,
        errors="coerce",
    )

    missing_count = int(feature_matrix.isna().sum().sum())
    if missing_count > 0:
        print(
            f"[WARNING] {missing_count} missing or nonnumeric feature values "
            "were replaced with 0."
        )
        feature_matrix = feature_matrix.fillna(0.0)

    feature_matrix = feature_matrix.astype(np.float32)

    labels = pd.to_numeric(dataframe["label"], errors="coerce")
    if labels.isna().any() or not set(labels.unique()).issubset({0, 1}):
        raise ValueError(
            "The 'label' column must contain only 0 and 1, where "
            "0 = LTS and 1 = HTS."
        )

    labels = labels.astype(int)

    if labels.nunique() != 2:
        raise ValueError("Both LTS (0) and HTS (1) classes must be present.")

    return feature_matrix, labels, feature_columns


def split_dataset(
    features: pd.DataFrame,
    labels: pd.Series,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.Series,
    pd.Series,
    pd.Series,
]:
    """Split the data into approximately equal training, validation, and test sets."""
    features_temp, features_test, labels_temp, labels_test = train_test_split(
        features,
        labels,
        test_size=1 / 3,
        stratify=labels,
        random_state=RANDOM_STATE,
    )

    features_train, features_validation, labels_train, labels_validation = (
        train_test_split(
            features_temp,
            labels_temp,
            test_size=0.5,
            stratify=labels_temp,
            random_state=RANDOM_STATE,
        )
    )

    return (
        features_train,
        features_validation,
        features_test,
        labels_train,
        labels_validation,
        labels_test,
    )


# =============================================================================
# Model tuning and fitting
# =============================================================================
def tune_random_forest(
    features_train: pd.DataFrame,
    labels_train: pd.Series,
    features_validation: pd.DataFrame,
    labels_validation: pd.Series,
) -> tuple[dict[str, Any], float, pd.DataFrame]:
    """Select random-forest hyperparameters by validation-set ROC AUC."""
    parameter_grid = {
        "n_estimators": [200, 500],
        "max_features": ["sqrt", "log2"],
        "max_depth": [None, 10, 30],
        "min_samples_leaf": [1, 2, 5],
        "class_weight": ["balanced", None],
    }

    best_validation_auc = -np.inf
    best_parameters: dict[str, Any] | None = None
    tuning_records: list[dict[str, Any]] = []

    print("[INFO] Starting random-forest hyperparameter search...")

    for parameters in ParameterGrid(parameter_grid):
        model = RandomForestClassifier(
            **parameters,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            bootstrap=True,
            oob_score=True,
        )
        model.fit(features_train, labels_train)

        validation_probability = model.predict_proba(features_validation)[:, 1]
        validation_auc = roc_auc_score(
            labels_validation,
            validation_probability,
        )

        tuning_records.append(
            {
                **parameters,
                "validation_roc_auc": validation_auc,
                "training_oob_score": model.oob_score_,
            }
        )

        if validation_auc > best_validation_auc:
            best_validation_auc = validation_auc
            best_parameters = dict(parameters)

    if best_parameters is None:
        raise RuntimeError("Random-forest hyperparameter tuning failed.")

    tuning_results = pd.DataFrame(tuning_records).sort_values(
        "validation_roc_auc",
        ascending=False,
    )

    return best_parameters, best_validation_auc, tuning_results


def fit_final_random_forest(
    features_train_validation: pd.DataFrame,
    labels_train_validation: pd.Series,
    best_parameters: dict[str, Any],
) -> RandomForestClassifier:
    """Refit the final model using the combined training and validation sets."""
    final_model = RandomForestClassifier(
        **best_parameters,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        bootstrap=True,
        oob_score=True,
    )
    final_model.fit(features_train_validation, labels_train_validation)
    return final_model


# =============================================================================
# SHAP compatibility and feature-importance plots
# =============================================================================
def extract_hts_shap_values(shap_values: Any) -> np.ndarray:
    """
    Extract SHAP values for the positive HTS class across SHAP versions.

    Supported formats include:
    - list[class_0_values, class_1_values]
    - array with shape (n_samples, n_features)
    - array with shape (n_samples, n_features, n_classes)
    """
    if isinstance(shap_values, list):
        return np.asarray(shap_values[POSITIVE_CLASS])

    shap_array = np.asarray(shap_values)

    if shap_array.ndim == 3:
        return shap_array[:, :, POSITIVE_CLASS]

    if shap_array.ndim != 2:
        raise ValueError(f"Unexpected SHAP array shape: {shap_array.shape}")

    return shap_array


def plot_top_shap_importance(
    shap_importance: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_FEATURES,
) -> None:
    """Plot the top genes ranked by mean absolute SHAP value."""
    plot_data = (
        shap_importance.sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .iloc[::-1]
        .copy()
    )

    figure_height = max(6, top_n * 0.42)
    fig, ax = plt.subplots(figsize=(8, figure_height))

    scatter = ax.scatter(
        plot_data["mean_abs_shap"],
        plot_data["gene"],
        c=plot_data["mean_abs_shap"],
        cmap="RdYlBu_r",
        s=90,
        edgecolors="none",
    )

    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {top_n} Genes by SHAP Importance")
    ax.grid(False)

    colorbar = fig.colorbar(scatter, ax=ax)
    colorbar.set_label("Mean |SHAP value|")

    fig.tight_layout()
    save_svg(fig, save_path)


def plot_rf_feature_importance(
    feature_importance: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_FEATURES,
) -> None:
    """Plot the top genes ranked by the built-in random-forest importance."""
    plot_data = (
        feature_importance.sort_values("importance", ascending=False)
        .head(top_n)
        .iloc[::-1]
        .copy()
    )

    figure_height = max(6, top_n * 0.42)
    fig, ax = plt.subplots(figsize=(8, figure_height))

    scatter = ax.scatter(
        plot_data["importance"],
        plot_data["gene"],
        c=plot_data["importance"],
        cmap="RdYlBu_r",
        s=90,
        edgecolors="none",
    )

    ax.set_xlabel("Random-forest feature importance")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {top_n} Random-Forest Feature Importances")
    ax.grid(False)

    colorbar = fig.colorbar(scatter, ax=ax)
    colorbar.set_label("Feature importance")

    fig.tight_layout()
    save_svg(fig, save_path)


def save_shap_summary_plot(
    shap_values: np.ndarray,
    feature_data: pd.DataFrame,
    feature_columns: list[str],
    save_path: Path,
) -> None:
    """Save a SHAP summary beeswarm plot."""
    import shap

    shap.summary_plot(
        shap_values,
        feature_data,
        feature_names=feature_columns,
        show=False,
        cmap="RdYlBu_r",
    )

    fig = plt.gcf()
    fig.set_size_inches(10, 8)
    save_svg(fig, save_path)


def run_shap_analysis(
    final_model: RandomForestClassifier,
    complete_features: pd.DataFrame,
    feature_columns: list[str],
    output_directory: Path,
    figure_directory: Path,
) -> None:
    """Explain the refitted final model and summarize SHAP importance over all cells."""
    try:
        import shap

        print(
            "[INFO] Calculating SHAP values for the refitted final random-forest "
            "model using the complete dataset..."
        )

        explainer = shap.TreeExplainer(final_model)
        raw_shap_values = explainer.shap_values(complete_features)
        hts_shap_values = extract_hts_shap_values(raw_shap_values)

        shap_importance = pd.DataFrame(
            {
                "gene": feature_columns,
                "mean_abs_shap": np.abs(hts_shap_values).mean(axis=0),
                "mean_shap": hts_shap_values.mean(axis=0),
            }
        ).sort_values("mean_abs_shap", ascending=False)

        shap_importance.to_csv(
            output_directory / "rf_shap_gene_importance_final_model_all_data.csv",
            index=False,
        )

        shap_importance.head(TOP_N_FEATURES).to_csv(
            output_directory
            / f"top{TOP_N_FEATURES}_shap_genes_by_abs_final_model_rf.csv",
            index=False,
        )

        plot_top_shap_importance(
            shap_importance,
            figure_directory
            / f"rf_top{TOP_N_FEATURES}_shap_gene_importance_final_model.svg",
        )

        save_shap_summary_plot(
            hts_shap_values,
            complete_features,
            feature_columns,
            figure_directory / "rf_shap_summary_final_model_all_data.svg",
        )

        print("[INFO] SHAP analysis completed successfully.")

    except ImportError as error:
        raise ImportError(
            "The SHAP package is required. Install it with: pip install shap"
        ) from error
    except Exception as error:
        raise RuntimeError(f"Random-forest SHAP analysis failed: {error}") from error


# =============================================================================
# Model evaluation and diagnostic plots
# =============================================================================
def calculate_metrics(
    true_labels: pd.Series,
    predicted_labels: np.ndarray,
    predicted_probabilities: np.ndarray,
) -> dict[str, float]:
    """Calculate binary-classification performance metrics."""
    return {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "f1": f1_score(true_labels, predicted_labels, zero_division=0),
        "roc_auc": roc_auc_score(true_labels, predicted_probabilities),
        "average_precision": average_precision_score(
            true_labels,
            predicted_probabilities,
        ),
    }


def save_confusion_matrix(
    confusion: np.ndarray,
    class_names: list[str],
    save_path: Path,
    title: str = "Confusion Matrix",
) -> None:
    """Save a confusion-matrix heatmap."""
    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    image = ax.imshow(confusion, interpolation="nearest", cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(confusion.shape[1]),
        yticks=np.arange(confusion.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True class",
        xlabel="Predicted class",
        title=title,
    )

    threshold = confusion.max() / 2 if confusion.max() > 0 else 0.5

    for row_index in range(confusion.shape[0]):
        for column_index in range(confusion.shape[1]):
            ax.text(
                column_index,
                row_index,
                format(confusion[row_index, column_index], "d"),
                ha="center",
                va="center",
                color=(
                    "white"
                    if confusion[row_index, column_index] > threshold
                    else "black"
                ),
            )

    fig.tight_layout()
    save_svg(fig, save_path)


def plot_roc_curve(
    true_labels: pd.Series,
    predicted_probabilities: np.ndarray,
    save_path: Path,
) -> None:
    """Save the test-set receiver operating characteristic curve."""
    auc = roc_auc_score(true_labels, predicted_probabilities)
    false_positive_rate, true_positive_rate, _ = roc_curve(
        true_labels,
        predicted_probabilities,
    )

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"ROC AUC = {auc:.4f}",
    )
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5)

    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Random-Forest Test ROC Curve")
    ax.legend(loc="lower right")

    fig.tight_layout()
    save_svg(fig, save_path)


def plot_precision_recall_curve(
    true_labels: pd.Series,
    predicted_probabilities: np.ndarray,
    save_path: Path,
) -> None:
    """Save the test-set precision-recall curve."""
    average_precision = average_precision_score(
        true_labels,
        predicted_probabilities,
    )
    precision, recall, _ = precision_recall_curve(
        true_labels,
        predicted_probabilities,
    )

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(
        recall,
        precision,
        linewidth=2,
        label=f"AP = {average_precision:.4f}",
    )

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Random-Forest Test Precision-Recall Curve")
    ax.legend(loc="lower left")

    fig.tight_layout()
    save_svg(fig, save_path)


def plot_prediction_score_distribution(
    true_labels: np.ndarray,
    predicted_probabilities: np.ndarray,
    save_path: Path,
) -> None:
    """Save the distribution of predicted HTS probabilities in the test set."""
    fig, ax = plt.subplots(figsize=(6.5, 5.5))

    ax.hist(
        predicted_probabilities[true_labels == NEGATIVE_CLASS],
        bins=30,
        alpha=0.7,
        label=f"{NEGATIVE_CLASS_NAME} ({NEGATIVE_CLASS})",
        edgecolor="black",
        color=LTS_COLOR,
    )
    ax.hist(
        predicted_probabilities[true_labels == POSITIVE_CLASS],
        bins=30,
        alpha=0.7,
        label=f"{POSITIVE_CLASS_NAME} ({POSITIVE_CLASS})",
        edgecolor="black",
        color=HTS_COLOR,
    )

    ax.set_xlabel(f"Predicted probability of {POSITIVE_CLASS_NAME}")
    ax.set_ylabel("Cell count")
    ax.set_title("Random-Forest Test Prediction-Score Distribution")
    ax.legend()

    fig.tight_layout()
    save_svg(fig, save_path)


def plot_oob_error_curve(
    base_model: RandomForestClassifier,
    features_train_validation: pd.DataFrame,
    labels_train_validation: pd.Series,
    save_path: Path,
) -> None:
    """Plot OOB error curves for sqrt and log2 max-feature settings."""
    candidate_max_features = ["sqrt", "log2"]
    colors = [LTS_COLOR, HTS_COLOR]
    labels = ["max_features = sqrt", "max_features = log2"]
    estimator_counts = list(range(10, 301, 10))

    fig, ax = plt.subplots(figsize=(7, 5))

    for index, max_features in enumerate(candidate_max_features):
        errors = []

        for estimator_count in estimator_counts:
            model = clone(base_model)
            model.set_params(
                n_estimators=estimator_count,
                max_features=max_features,
                oob_score=True,
                bootstrap=True,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
            model.fit(features_train_validation, labels_train_validation)
            errors.append(1 - model.oob_score_)

        ax.plot(
            estimator_counts,
            errors,
            label=labels[index],
            color=colors[index],
            linewidth=2.5,
        )

    ax.set_xlabel("Number of trees")
    ax.set_ylabel("OOB error rate")
    ax.set_title("Random-Forest OOB Error Curve")
    ax.legend(frameon=False)

    fig.tight_layout()
    save_svg(fig, save_path)


# =============================================================================
# Main workflow
# =============================================================================
def main() -> None:
    code_directory = get_code_directory()
    data_path = code_directory / "df_expr.csv"
    output_directory = code_directory / "rf_hts_lts_results"
    figure_directory = output_directory / "figures"

    ensure_directory(output_directory)
    ensure_directory(figure_directory)

    print(f"[INFO] Working directory: {code_directory}")
    print(f"[INFO] Input data file: {data_path}")

    features, labels, feature_columns = load_expression_data(data_path)

    print(f"[INFO] Number of observations: {features.shape[0]}")
    print(f"[INFO] Number of features: {features.shape[1]}")
    print(
        f"[INFO] Class distribution: "
        f"{NEGATIVE_CLASS_NAME} (0) = {(labels == 0).sum()}, "
        f"{POSITIVE_CLASS_NAME} (1) = {(labels == 1).sum()}"
    )

    (
        features_train,
        features_validation,
        features_test,
        labels_train,
        labels_validation,
        labels_test,
    ) = split_dataset(features, labels)

    print(f"[INFO] Training-set size: {features_train.shape[0]}")
    print(f"[INFO] Validation-set size: {features_validation.shape[0]}")
    print(f"[INFO] Test-set size: {features_test.shape[0]}")

    best_parameters, best_validation_auc, tuning_results = tune_random_forest(
        features_train,
        labels_train,
        features_validation,
        labels_validation,
    )

    tuning_results.to_csv(
        output_directory / "rf_validation_tuning_results.csv",
        index=False,
    )

    print(f"[INFO] Best random-forest parameters: {best_parameters}")
    print(f"[INFO] Best validation ROC AUC: {best_validation_auc:.4f}")

    with open(
        output_directory / "best_rf_params.json",
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            {
                "best_parameters": best_parameters,
                "validation_roc_auc": best_validation_auc,
            },
            file_handle,
            indent=4,
            ensure_ascii=True,
        )

    features_train_validation = pd.concat(
        [features_train, features_validation],
        axis=0,
    )
    labels_train_validation = pd.concat(
        [labels_train, labels_validation],
        axis=0,
    )

    print(
        "[INFO] Refitting the final random-forest model using the combined "
        "training and validation sets..."
    )
    final_model = fit_final_random_forest(
        features_train_validation,
        labels_train_validation,
        best_parameters,
    )
    print(f"[INFO] Final random-forest OOB score: {final_model.oob_score_:.4f}")

    plot_oob_error_curve(
        final_model,
        features_train_validation,
        labels_train_validation,
        figure_directory / "rf_oob_error_curve.svg",
    )

    print("[INFO] Evaluating the final model on the independent test set...")
    test_probabilities = final_model.predict_proba(features_test)[:, 1]
    test_predictions = (test_probabilities >= 0.5).astype(int)

    test_predictions_table = pd.DataFrame(
        {
            "true_label": labels_test.values,
            "true_class": labels_test.map(
                {NEGATIVE_CLASS: NEGATIVE_CLASS_NAME, POSITIVE_CLASS: POSITIVE_CLASS_NAME}
            ).values,
            "predicted_label": test_predictions,
            "predicted_class": pd.Series(test_predictions).map(
                {NEGATIVE_CLASS: NEGATIVE_CLASS_NAME, POSITIVE_CLASS: POSITIVE_CLASS_NAME}
            ).values,
            "predicted_probability_HTS": test_probabilities,
        }
    )
    test_predictions_table.to_csv(
        output_directory / "rf_test_predictions.csv",
        index=False,
    )

    test_metrics = calculate_metrics(
        labels_test,
        test_predictions,
        test_probabilities,
    )
    pd.DataFrame([test_metrics]).to_csv(
        output_directory / "rf_test_metrics.csv",
        index=False,
    )

    print("[INFO] Random-forest test metrics:")
    for metric_name, metric_value in test_metrics.items():
        print(f"[RESULT] {metric_name}: {metric_value:.4f}")

    test_confusion_matrix = confusion_matrix(labels_test, test_predictions)
    save_confusion_matrix(
        test_confusion_matrix,
        [NEGATIVE_CLASS_NAME, POSITIVE_CLASS_NAME],
        figure_directory / "rf_test_confusion_matrix.svg",
        title="Random-Forest Test Confusion Matrix",
    )
    plot_roc_curve(
        labels_test,
        test_probabilities,
        figure_directory / "rf_test_roc_curve.svg",
    )
    plot_precision_recall_curve(
        labels_test,
        test_probabilities,
        figure_directory / "rf_test_precision_recall_curve.svg",
    )
    plot_prediction_score_distribution(
        labels_test.to_numpy(),
        test_probabilities,
        figure_directory / "rf_test_prediction_score_distribution.svg",
    )

    feature_importance = pd.DataFrame(
        {
            "gene": feature_columns,
            "importance": final_model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    feature_importance.to_csv(
        output_directory / "rf_feature_importance.csv",
        index=False,
    )
    plot_rf_feature_importance(
        feature_importance,
        figure_directory
        / f"rf_top{TOP_N_FEATURES}_built_in_feature_importance.svg",
    )

    run_shap_analysis(
        final_model,
        features,
        feature_columns,
        output_directory,
        figure_directory,
    )

    print("[INFO] Analysis completed successfully.")
    print(f"[INFO] Results directory: {output_directory}")
    print(f"[INFO] Figure directory: {figure_directory}")


if __name__ == "__main__":
    main()
